# broadcasting-rules composite — cx27: reduce then broadcast — per-row normalization

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `einops-reduce`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "broadcasting-rules"
DD_ATOM_IDS = ["broadcasting-rules", "einops-reduce"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "Einops: Reduce"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Normalization is the canonical reduce-then-broadcast pattern: you `reduce` to compute per-sample statistics (mean, std), then rely on **broadcasting rules** to subtract / divide them back across the original tensor without ever expanding the stats by hand.

The trick: use `reduce(..., 'b d -> b 1', 'mean')` (keepdim semantics via the explicit `1`). The trailing `1` axis is exactly what broadcasting needs to align against the original `(b, d)` tensor — a missing axis would force you to `unsqueeze` manually; a fully-collapsed `(b,)` would broadcast to the WRONG axis. `(b, 1)` is the keepdim sweet spot that makes broadcasting automatic.

### Composite Exercise — reduce then broadcast — per-row normalization

**Atoms exercised together**: `broadcasting-rules`, `einops-reduce`

Implement `cx27_row_normalize(x, eps)` that returns the per-row z-score normalized version of an `(B, D)` matrix:

1. **Reduce** to per-row mean and std with the `'b d -> b 1'` pattern (KEEP the trailing 1 — that's what makes broadcasting auto-align).
2. **Broadcast** the subtraction and division back across the original tensor: `(x - mu) / (sigma + eps)`. Do not call `.expand` / `.repeat` — broadcasting rules handle it for free.

Return shape `(B, D)`. The result should have per-row mean ~0 and std ~1.

Cross-check against `(x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True, unbiased=False) + eps)`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_row_normalize(x, eps=1e-5):
    raise NotImplementedError

def _test_cx27():
    # Case A: standard (B, D) matrix.
    x = t.randn(4, 16) * 5 + 3  # off-center, off-scale
    out = cx27_row_normalize(x, eps=1e-5)
    assert tuple(out.shape) == tuple(x.shape), f'shape changed: {tuple(out.shape)}'
    # Per-row mean ~ 0.
    row_means = out.mean(dim=1)
    assert t.allclose(row_means, t.zeros(4), atol=1e-4), f'row means not ~0: {row_means}'
    # Per-row std ~ 1.
    row_stds = out.std(dim=1, unbiased=False)
    assert t.allclose(row_stds, t.ones(4), atol=1e-3), f'row stds not ~1: {row_stds}'

    # Cross-check against the torch keepdim formulation.
    ref = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True, unbiased=False) + 1e-5)
    assert t.allclose(out, ref, atol=1e-5), 'normalized values diverge from keepdim reference'

    # Case B: wider D.
    x2 = t.randn(8, 128)
    out2 = cx27_row_normalize(x2, eps=1e-5)
    assert tuple(out2.shape) == (8, 128)
    assert t.allclose(out2.mean(dim=1), t.zeros(8), atol=1e-4)

    # Case C: eps actually prevents zero-division on a constant row.
    x3 = t.ones(2, 5)  # std = 0
    out3 = cx27_row_normalize(x3, eps=1e-3)
    assert t.isfinite(out3).all(), 'eps should prevent inf/nan on constant rows'
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_row_normalize(x, eps=1e-5):
    # Atom A (einops-reduce): keep the trailing 1 axis so broadcasting auto-aligns.
    mu = reduce(x, 'b d -> b 1', 'mean')
    # variance via reduce of (x - mu) ** 2 — still keeping the trailing 1.
    var = reduce((x - mu) ** 2, 'b d -> b 1', 'mean')
    sigma = (var + 0.0).sqrt()
    # Atom B (broadcasting-rules): (b, 1) aligns against (b, d) without manual expand.
    return (x - mu) / (sigma + eps)
```

The trailing `1` in the reduce pattern is doing all the broadcasting work. Without it (`'b d -> b'`), the result would be shape `(B,)` and broadcasting would align it against the LAST axis of `x` (D), giving you per-COLUMN normalization instead. With it, broadcasting aligns `(B, 1)` against `(B, D)` and stretches the 1-axis automatically — no explicit expand or unsqueeze required. This is the einops-as-keepdim idiom.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["Numpy: Vectorization and broadcasting", "Einops: Reduce"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()